In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from scipy.optimize import minimize

from rich import print
import seaborn as sns
sns.set()

import ewatercycle
import ewatercycle.models
import ewatercycle.forcing
from scipy.stats import qmc
from ipywidgets import IntProgress

In [2]:
calibration_start_date = "1975-01-01"
calibration_end_date = "2010-12-31"

In [3]:
forcing_path_ERA5 = Path.home() / "BEP-beau/BEP/code" / "CatchmentArea" / "ERA5_2" / "own_shapefile_2"
load_location = forcing_path_ERA5 / "work" / "diagnostic" / "script"  
ERA5_forcing = ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=load_location)

data = pd.read_csv("mohembo_daily_water_discharge_data.csv", index_col='date', parse_dates=True, dayfirst=True)
data_daily = data.resample('D').interpolate()
data_daily.columns = ['Discharge (m^3/s)']

In [4]:
Area_km2 = 173696.852

def mmday_to_m3s(mmday_data, area):
    return (mmday_data * area) / 86.4

In [13]:
def RMSE(modelled, observed, start, end):
    start = pd.to_datetime(start).tz_localize(None)
    end = pd.to_datetime(end).tz_localize(None)
    
    modelled.index = pd.to_datetime(modelled.index)
    observed.index = pd.to_datetime(observed.index)

    df = pd.concat([modelled.reindex(observed.index, method='ffill'), observed], axis=1, keys=['Modelled', 'Observed'])
    df = df.dropna()
    df = df[(df.index > start) & (df.index < end)]

    n = len(df)
    a = (df['Observed'] - df['Modelled'])**2
    a1 = a.sum()
    rmse = np.sqrt(a1 / n)
    return rmse

In [20]:
N = 2000
s_0 = np.array([0,  100,  0,  5,  0])

param_names = ["Imax", "Ce", "Sumax", "Beta", "Pmax", "Tlag", "Kf", "Ks", "FM"]
param_mins = np.array([0, 0.2, 250, 0.5, 0.001, 1, 0.0005, 0.00005, 0.00001])
param_maxs = np.array([8, 1.5, 2500, 4, 0.3, 90, 0.1, 0.002, 1])

sampler = qmc.LatinHypercube(d=len(param_names))
sample = sampler.random(n=N)
parameters = qmc.scale(sample, param_mins, param_maxs)

In [21]:
ensemble = []

for counter in range(N): 
    ensemble.append(ewatercycle.models.HBVLocal(forcing=ERA5_forcing))
    config_file, _ = ensemble[counter].setup(parameters = parameters[counter],  initial_storage=s_0)
    ensemble[counter].initialize(config_file)

In [22]:
f = IntProgress(min=0, max=N)
display(f)

RMSE_values = []
for ensembleMember in ensemble:
    Q_m_RMSE = []
    time_RMSE = []
    while ensembleMember.time < ensembleMember.end_time:
        ensembleMember.update()
        discharge_this_timestep = ensembleMember.get_value("Q")
        Q_m_RMSE.append(discharge_this_timestep[0])
        time_RMSE.append(ensembleMember.time_as_datetime)

    Q_m_RMSE = mmday_to_m3s(np.array(Q_m_RMSE), Area_km2)
    discharge_dataframe = pd.DataFrame({'model output': Q_m_RMSE}, index=pd.to_datetime(time_RMSE))

    rmsevalue = RMSE(discharge_dataframe['model output'], data_daily['Discharge (m^3/s)'], calibration_start_date, calibration_end_date)
    RMSE_values.append(rmsevalue)
    
    del Q_m_RMSE, time_RMSE, discharge_dataframe, rmsevalue
    f.value += 1
    
for ensembleMember in ensemble:
    ensembleMember.finalize()

IntProgress(value=0, max=2000)

In [27]:
best_index = np.argmin(RMSE_values)
best_parameters = parameters[best_index]
best_RMSE = RMSE_values[best_index]
print(list(zip(param_names, np.round(best_parameters, decimals=3))))
print(f"Best RMSE: {best_RMSE:.3f}")
best_parameters

[
    ('Imax', 5.242),
    ('Ce', 0.594),
    ('Sumax', 2411.181),
    ('Beta', 3.518),
    ('Pmax', 0.167),
    ('Tlag', 34.311),
    ('Kf', 0.019),
    ('Ks', 0.001),
    ('FM', 0.377)
]

Best RMSE: 76.045

array([5.24184140e+00, 5.93982257e-01, 2.41118070e+03, 3.51779378e+00,
       1.66585195e-01, 3.43109685e+01, 1.92236149e-02, 1.01446811e-03,
       3.77397207e-01])